In [1]:
import heapq

# -------------------------------
# Trie Data Structure
# -------------------------------
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end_of_word = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for char in word.lower():
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
        node.is_end_of_word = True

    def search_prefix(self, prefix):
        node = self.root
        for char in prefix.lower():
            if char not in node.children:
                return []
            node = node.children[char]
        return self._collect_words(node, prefix.lower())

    def _collect_words(self, node, prefix):
        results = []
        if node.is_end_of_word:
            results.append(prefix.title())
        for char, child in node.children.items():
            results.extend(self._collect_words(child, prefix + char))
        return results


# -------------------------------
# A* Search Algorithm
# -------------------------------
class Graph:
    def __init__(self):
        self.edges = {}
        self.distances = {}

    def add_edge(self, from_node, to_node, distance):
        self.edges.setdefault(from_node, []).append(to_node)
        self.edges.setdefault(to_node, []).append(from_node)
        self.distances[(from_node, to_node)] = distance
        self.distances[(to_node, from_node)] = distance

def heuristic(node, goal):
    return abs(len(node) - len(goal))  # toy heuristic

def a_star_search(graph, start, goal):
    if start not in graph.edges or goal not in graph.edges:
        return None

    open_set = []
    heapq.heappush(open_set, (0, start))
    came_from = {}
    g_score = {node: float('inf') for node in graph.edges}
    g_score[start] = 0
    f_score = {node: float('inf') for node in graph.edges}
    f_score[start] = heuristic(start, goal)

    while open_set:
        _, current = heapq.heappop(open_set)
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]

        for neighbor in graph.edges[current]:
            tentative_g = g_score[current] + graph.distances[(current, neighbor)]
            if tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f_score[neighbor] = tentative_g + heuristic(neighbor, goal)
                heapq.heappush(open_set, (f_score[neighbor], neighbor))
    return None


# -------------------------------
# Interactive Example with Numbered Selection
# -------------------------------
if __name__ == "__main__":
    # Build Trie with MTR stations
    stations = ["Sha Tin", "Shau Kei Wan", "Central", "Admiralty", "Mong Kok", "Yuen Long"]
    trie = Trie()
    for station in stations:
        trie.insert(station)

    # Build Graph with simplified routes
    g = Graph()
    g.add_edge("Sha Tin", "Admiralty", 20)
    g.add_edge("Admiralty", "Central", 5)
    g.add_edge("Central", "Mong Kok", 15)
    g.add_edge("Mong Kok", "Yuen Long", 25)

    # Autocomplete input
    prefix = input("Enter station prefix: ")
    suggestions = trie.search_prefix(prefix)
    if suggestions:
        print("Suggestions:")
        for i, s in enumerate(suggestions, 1):
            print(f"{i}. {s}")
        choice = int(input("Select start station (number): "))
        start = suggestions[choice - 1]
    else:
        print("No station found.")
        exit()

    # Destination input
    prefix_goal = input("Enter destination prefix: ")
    suggestions_goal = trie.search_prefix(prefix_goal)
    if suggestions_goal:
        print("Suggestions:")
        for i, s in enumerate(suggestions_goal, 1):
            print(f"{i}. {s}")
        choice_goal = int(input("Select destination station (number): "))
        goal = suggestions_goal[choice_goal - 1]
    else:
        print("No station found.")
        exit()

    # Route planning
    route = a_star_search(g, start, goal)
    if route:
        print("Shortest route:", " → ".join(route))
    else:
        print("No route found between those stations.")


Enter station prefix:  sha


Suggestions:
1. Sha Tin
2. Shau Kei Wan


Select start station (number):  1
Enter destination prefix:  yuen


Suggestions:
1. Yuen Long


Select destination station (number):  1


Shortest route: Sha Tin → Admiralty → Central → Mong Kok → Yuen Long
